In [48]:
import geopandas as gpd
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text, inspect

In [49]:
PATH = Path(f"C:\\Users\\lschild\\Documents\\SOLVE")
data = PATH / "dummy_data"


In [50]:
def conn_db():
    host = "localhost"
    port = 5432
    database = "SOLVE"
    user ="postgres"
    password = "postgres"

    engine = create_engine(f"postgresql+psycopg://{user}:{password}@{host}:{port}/{database}")
    with engine.connect() as connection:
        print("Verbindung erfolgreich")
    return engine

In [51]:
waterbody_shp = data / "lakewaterbody.shp"
stammdaten_csv = data / "stammdaten.csv"
gdf = gpd.read_file(waterbody_shp)
stammdaten = pd.read_csv(stammdaten_csv)


In [52]:
from shapely.geometry import Polygon, MultiPolygon


def as_multipolygon(geometry):
    if geometry is None:
        return None

    if geometry.geom_type == "MultiPolygon":
        return geometry

    if geometry.geom_type == "Polygon":
        return MultiPolygon([geometry])

In [53]:
def update_db(
    element,
    engine,
    name,
    primary_key,
    geo=False,
    schema="water_kg"
):

    if isinstance(primary_key, str):
        primary_key = [primary_key]

    element = element.drop_duplicates(
        subset=primary_key
    ).copy()

    with engine.begin() as connection:

        inspector = inspect(connection)

        table_exists = inspector.has_table(
            name,
            schema=schema
        )

        if not table_exists:
            if geo:
                element.to_postgis(
                    name=name,
                    con=connection,
                    schema=schema,
                    if_exists="replace",
                    index=False
                )
            else:
                element.to_sql(
                    name=name,
                    con=connection,
                    schema=schema,
                    if_exists="replace",
                    index=False
                )

            return

        existing = pd.read_sql_table(
            name,
            con=connection,
            schema=schema
        )

        existing_keys = existing[
            primary_key
        ].drop_duplicates()

        new_rows = element.merge(
            existing_keys,
            on=primary_key,
            how="left",
            indicator=True
        )

        new_rows = new_rows[
            new_rows["_merge"] == "left_only"
        ].drop(columns="_merge")

        if new_rows.empty:
            print(f"No new rows for {schema}.{name}")
            return

        if geo:
            new_rows.to_postgis(
                name=name,
                con=connection,
                schema=schema,
                if_exists="append",
                index=False
            )
        else:
            new_rows.to_sql(
                name=name,
                con=connection,
                schema=schema,
                if_exists="append",
                index=False
            )

        print(
            f"{len(new_rows)} new rows inserted into "
            f"{schema}.{name}"
        )

In [54]:
#get waterbody data from stammdaten.csv and lakewaterbody.shp
def load_waterbody(gdf, stammdaten):
    engine = conn_db()
    waterbody = gpd.GeoDataFrame(
        {
            "eu_cd_lw": gdf["EU_CD_LW"],
            "eu_cd_ls": gdf["EU_CD_LS"],
            "water_body_type": "lake",
            "name": gdf["S_NAME"],
            "lawa_id": stammdaten["LAWA_ID"],
            "area_km2": stammdaten["FLAECHE_ATKIS"],
            "volume_m3": stammdaten["VOLUMEN"],
            "max_depth_m": stammdaten["Z_MAX"],
            "artificial": stammdaten["ARTIFICIAL_2021"],
            "modified": stammdaten["MODIFIED_2021"],
            "river_basin_code": gdf["RBD_CD"],
            "planning_unit_code": gdf["PLANU_CD"],
            "water_type": stammdaten["TYP_2021"],
        },
        geometry=gdf.geometry,
        crs=gdf.crs,
    )

    # interne Duplikate in den neuen Daten entfernen
    update_db(waterbody, engine, "water_body", "eu_cd_lw", geo  = True)
    return waterbody


In [55]:
groundwaterbody_shp = data / "groundwaterbody.shp"
gwb = gpd.read_file(groundwaterbody_shp)

In [56]:
#get waterbody data from groundwaterbody.shp
def load_groundwaterbody(gwb):
    engine = conn_db()
    groundwaterbody = gpd.GeoDataFrame(
        {
            "eu_cd_gb": gwb["EU_CD_GB"],
            "name": gwb["NAMETEXT"],
            "horizon": gwb["HORIZON"].astype(int),
            "aquifer_type": gwb["AQUI_TYPE"],
            "geological_formation": gwb["GEOL_FORM"],
            "layered": gwb["LAYERED"],
            "river_basin_code": gwb["RBD_CD"],
            "planning_unit_code": gwb["PLANU_CD"],
            "quantitative_status": gwb["QUANT_STAT"],
            "chemical_status": gwb["CHEM_STAT"],
            "metadata_reference": gwb["METADATA"],
        },
        geometry=gwb.geometry,
        crs=gwb.crs,
    )
    

    update_db(groundwaterbody, engine, "groundwater_body","eu_cd_gb")
    return groundwaterbody

In [57]:
waterbody = load_waterbody(gdf, stammdaten)
groundwaterbody = load_groundwaterbody(gwb)

Verbindung erfolgreich
No new rows for water_kg.water_body


c:\Users\lschild\Documents\SOLVE\.venv\Lib\site-packages\pandas\io\sql.py:1747: SAWarning: Did not recognize type 'geometry' of column 'geometry'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


Verbindung erfolgreich
No new rows for water_kg.groundwater_body


c:\Users\lschild\Documents\SOLVE\.venv\Lib\site-packages\pandas\io\sql.py:1747: SAWarning: Did not recognize type 'geometry' of column 'geometry'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


In [58]:
source = gpd.read_file(data / "catchments.shp").to_crs(25833)

In [59]:
# Einzugsgebiete aus catchments.shp
def load_catchments(source):
    engine = conn_db()
    
    catchments = gpd.GeoDataFrame(
        {
            "catchment_id": source["DRAIN_C"],
            "name": source["DRAIN_N"],
            "area_km2": pd.to_numeric(source["AREA_CA"], errors="coerce"),
            "area_number": source["AREA_NO"],
            "water_body_id": source["EU_CD_W"],
            "planning_unit_code": source["PLANU_C"],
            "river_basin_code": source["RBD_CD"],
            "water_authority": source["WA_CD"],
            "description_from": source["DESCR_F"],
            "description_to": source["DESCR_T"],
            "comment": source["COMMENT"],
            "metadata_reference": source["METADATA"],
        },
        geometry=source.geometry, crs=source.crs,
    )

    update_db(catchments, engine, "catchment","catchment_id", geo = True)  
    

# Messstellen aus messstellen.shp


In [60]:
load_catchments(source)

Verbindung erfolgreich
No new rows for water_kg.catchment


c:\Users\lschild\Documents\SOLVE\.venv\Lib\site-packages\pandas\io\sql.py:1747: SAWarning: Did not recognize type 'geometry' of column 'geometry'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


In [61]:
source = data / "messstellen.shp"
source = gpd.read_file(source)

In [62]:
def load_monitoring_stations(source):
    engine = conn_db()
    stations = gpd.GeoDataFrame(
        {
            "station_code": source["EU_CD_SM"],
            "name": source["NAME_STN"],
            "water_body_id": source["EU_CD_WB"],
        },
        geometry=source.geometry, crs=source.crs,
    )
    stations = stations.rename_geometry("location")
    update_db(stations, engine, "monitoring_station","station_code")

In [63]:
load_monitoring_stations(source)

Verbindung erfolgreich
No new rows for water_kg.monitoring_station


c:\Users\lschild\Documents\SOLVE\.venv\Lib\site-packages\pandas\io\sql.py:1747: SAWarning: Did not recognize type 'geometry' of column 'location'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


In [64]:
bio = pd.read_csv(data / "einzeldaten.csv", low_memory=False)
chem = pd.read_csv(data / "einzeldaten_chemie.csv", low_memory=False)

In [65]:
def load_parameters(bio, chem):
    engine = conn_db()

    bio_param = bio.rename(columns={
        "PARAMETER": "name",
        "QK": "quality_component",
        "EINHEIT": "unit",
    })[
        ["name", "quality_component", "unit"]
    ].copy()

    chem_param = chem.rename(columns={
        "Parameterbezeichnung": "name",
        "QK": "quality_component",
        "EINHEIT": "unit",
    })[
        ["name", "quality_component", "unit"]
    ].copy()

    parameter = pd.concat(
        [bio_param, chem_param],
        ignore_index=True
    )

    parameter = parameter.drop_duplicates(
        subset=["name", "quality_component", "unit"]
    )

    with engine.begin() as connection:
        inspector = inspect(connection)

        table_exists = inspector.has_table(
            "parameter",
            schema="water_kg"
        )

        if not table_exists:
            parameter.to_sql(
                name="parameter",
                con=connection,
                schema="water_kg",
                if_exists="replace",
                index=False,
            )

            print(f"{len(parameter)} parameters inserted.")
            return parameter

        existing = pd.read_sql_table(
            "parameter",
            con=connection,
            schema="water_kg",
        )

        new_rows = parameter.merge(
            existing[
                ["name", "quality_component", "unit"]
            ],
            on=["name", "quality_component", "unit"],
            how="left",
            indicator=True,
        )

        new_rows = new_rows[
            new_rows["_merge"] == "left_only"
        ].drop(columns="_merge")

        if not new_rows.empty:
            new_rows.to_sql(
                name="parameter",
                con=connection,
                schema="water_kg",
                if_exists="append",
                index=False,
            )

            print(f"{len(new_rows)} new parameters inserted.")
        else:
            print("No new parameters found.")

    return parameter

In [66]:
load_parameters(bio, chem)

Verbindung erfolgreich
No new parameters found.


,name,quality_component,unit
0,Achnanthes clevei var. rostrata,Diatomeen,NaN
8,Achnanthes,Diatomeen,NaN
10,Achnanthes ploenensis var. gessneri,Diatomeen,NaN
12,Achnanthidium caledonicum,Diatomeen,%
13,Achnanthidium caledonicum,Diatomeen,NaN
...,...,...,...
15519,Trockengewicht,Chemie Sedimente,mg/kg
18124,Phaeophytin,Chemie (See und FG),mg/l
37736,pH,Chemie (See und FG),NaN
96493,Siliziumoxid-Si,Chemie (See und FG),µg/l


In [67]:
source = pd.read_csv(data / "einzeldaten_abschnitte.csv", low_memory=False)

In [71]:
def station_ids():
    engine = conn_db()
    stations = pd.read_sql_table("monitoring_station", engine, schema="water_kg")
    result = {}
    for row in stations.itertuples():
        result[(str(row.station_code), row.water_body_id)] = row.station_id
        if row.name is not None:
            result[(str(row.name), row.water_body_id)] = row.station_id
    return result

# Probenabschnitte aus einzeldaten_abschnitte.csv
def load_sampling_sections(source):
    engine = conn_db()
    lookup = station_ids()
    sections = pd.DataFrame({
        "water_body_id": source["EU_CD_LW"],
        "source_station_code": (source["MESSSTELLE"]),
        "section_code": source["ABSCHNITT"],
        "sampling_year": pd.to_numeric(source["JAHR"], errors="coerce").astype("Int64"),
        "length_from": pd.to_numeric(source["LAENGE_VON"], errors="coerce"),
        "length_to": pd.to_numeric(source["LAENGE_BIS"], errors="coerce"),
        "depth_from": pd.to_numeric(source["TIEFE_VON"], errors="coerce"),
        "depth_to": pd.to_numeric(source["TIEFE_BIS"], errors="coerce"),
    })
    sections["station_id"] = sections.apply(
        lambda row: lookup.get((row["source_station_code"], row["water_body_id"])), axis=1
    )
    update_db(sections, engine, "sampling_section",["water_body_id", "source_station_code", "section_code", "sampling_year"],geo=False)

In [72]:
engine = conn_db()
stations = pd.read_sql_table(
    "monitoring_station",
    engine,
    schema="water_kg"
)

print(stations.columns.tolist())
print(stations.head())

Verbindung erfolgreich
['station_code', 'name', 'water_body_id', 'location', 'station_id']
             station_code            name         water_body_id  \
0            DESM_BB_2796            2796  DELW_DEBB80001581473   
1            DESM_BB_2794            2794  DELW_DEBB80001581473   
2            DESM_BB_2795            2795  DELW_DEBB80001581473   
3            DESM_BB_2792            2792  DELW_DEBB80001581473   
4  DESM_BB_80001581473_HM  80001581473_HM  DELW_DEBB80001581473   

                                            location  station_id  
0  0101000020E9640000000000009075184100000000FB74...           1  
1  0101000020E964000000000000D88C1841000000C00676...           2  
2  0101000020E964000000000000D4811841000000C07C75...           3  
3  0101000020E96400000000000090791841000000406275...           4  
4  0101000020E964000000000000DC801841000000009F75...           5  


c:\Users\lschild\Documents\SOLVE\.venv\Lib\site-packages\pandas\io\sql.py:1747: SAWarning: Did not recognize type 'geometry' of column 'location'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


In [73]:
load_sampling_sections(source)

Verbindung erfolgreich
Verbindung erfolgreich


c:\Users\lschild\Documents\SOLVE\.venv\Lib\site-packages\pandas\io\sql.py:1747: SAWarning: Did not recognize type 'geometry' of column 'location'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


In [74]:
bio = pd.read_csv(data / "einzeldaten.csv", low_memory=False)
chem = pd.read_csv(data / "einzeldaten_chemie.csv", low_memory=False)

In [75]:
# Biologische und chemische Einzelmessungen
def load_measurements(bio, chem):
    engine = conn_db()
    parameters = pd.read_sql_table("parameter", engine, schema="water_kg")
    parameter_lookup = {
        (str(row.name), None if pd.isna(row.quality_component) else str(row.quality_component),
         None if pd.isna(row.unit) else str(row.unit)): row.parameter_id
        for row in parameters.itertuples()
    }
    stations = station_ids()

    def convert(source, parameter_column, comment_column, biological=False):
        result = pd.DataFrame({
            "water_body_id": source["EU_CD_LW"],
            "source_station_code": source["MESSSTELLE"],
            "sampling_section": source["ID_ABSCHNITT"],
            "measured_at": pd.to_datetime(source["DATUM"], errors="coerce").dt.date,
            "value": pd.to_numeric(source["WERT"], errors="coerce"),
            "comment": source[comment_column],
            "method": source["METHODE"] if "METHODE" in source else None,
            "monitoring_program": source["MESSPROGRAMM"],
            "abundance_type": source["ABUNDANZART"] if biological else None,
        })
        result["station_id"] = result.apply(
            lambda row: stations.get((row["source_station_code"], row["water_body_id"])), axis=1
        )
        result["parameter_id"] = source.apply(
            lambda row: parameter_lookup.get((
                str(row[parameter_column]),
                None if pd.isna(row["QK"]) else str(row["QK"]),
                None if pd.isna(row["EINHEIT"]) else str(row["EINHEIT"]),
            )), axis=1,
        )
        return result.dropna(subset=["water_body_id", "parameter_id"])

    
    measurements = pd.concat([
        convert(bio, "PARAMETER", "KOMMENTAR", biological=True),
        convert(chem, "Parameterbezeichnung", "Kommentar"),
    ], ignore_index=True)
    return update_db(
        measurements,engine, "measurement",
        ["water_body_id", "source_station_code", "parameter_id", "measured_at",
         "sampling_section", "value"],
    )

In [77]:
load_measurements(bio, chem)

Verbindung erfolgreich
Verbindung erfolgreich


c:\Users\lschild\Documents\SOLVE\.venv\Lib\site-packages\pandas\io\sql.py:1747: SAWarning: Did not recognize type 'geometry' of column 'location'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


In [78]:
source = pd.read_csv(data / "messstellen_bewertung.csv", low_memory=False)


In [79]:
# Bewertungen von Wasserkoerpern und Messstellen
def load_assessments(source):
    engine = conn_db()
    body_source = pd.read_csv(data / "wasserkoerper_bewertung.csv", low_memory=False)
    body = pd.DataFrame({
        "water_body_id": body_source["EU_CD_LW"],
        "parameter": body_source["PARAMETER"],
        "assessment_year": pd.to_numeric(body_source["JAHR"], errors="coerce").astype("Int64"),
        "value": pd.to_numeric(body_source["WERT"], errors="coerce"),
        "method": body_source["METHODE"], 
        "quality_component": body_source["QK"],
    }).dropna(subset=["water_body_id", "parameter"])
    update_db(
        body,engine, "water_body_assessment",
        ["water_body_id", "parameter", "assessment_year", "method"],
    )

    lookup = station_ids()
    station = pd.DataFrame({
        "water_body_id": source["EU_CD_LW"],
        "source_station_code": source["MESSSTELLE"],
        "parameter": source["PARAMETER"],
        "assessment_year": pd.to_numeric(source["JAHR"], errors="coerce").astype("Int64"),
        "value": pd.to_numeric(source["WERT"], errors="coerce"),
        "method": source["METHODE"], 
        "quality_component": source["KOMPONENTE"],
        "unit": source["EINHEIT"],
        "description": source["BESCHREIBUNG"],
        "expert_judgement": source["EXPERTENURTEIL"],
        "comment": source["KOMMENTAR"],
        "reported_x": pd.to_numeric(source["X_WERT"], errors="coerce"),
        "reported_y": pd.to_numeric(source["Y_WERT"], errors="coerce"),
    })
    station["station_id"] = station.apply(
        lambda row: lookup.get((row["source_station_code"], row["water_body_id"])), axis=1
    )
    station = station.dropna(subset=["water_body_id", "parameter"])
    update_db(
        station,engine, "station_assessment",
        ["water_body_id", "source_station_code", "parameter", "assessment_year", "method"],
    )
    return body, station

In [80]:
load_assessments(source)

Verbindung erfolgreich
Verbindung erfolgreich


c:\Users\lschild\Documents\SOLVE\.venv\Lib\site-packages\pandas\io\sql.py:1747: SAWarning: Did not recognize type 'geometry' of column 'location'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


(            water_body_id                                          parameter  \
 0    DELW_DEBB80001581473  Teilbewertung Diatomeen/Zustands-/Potentialkla...   
 1    DELW_DEBB80001581473                               Ökologischer Zustand   
 2    DELW_DEBB80001581473                            Bewertung Phytoplankton   
 3    DELW_DEBB80001581473                    Bewertung Makrophyten/Diatomeen   
 4    DELW_DEBB80001582739                          Teilbewertung Makrophyten   
 5    DELW_DEBB80001582739                               Ökologischer Zustand   
 6    DELW_DEBB80001582739                            Bewertung Phytoplankton   
 7    DELW_DEBB80001582739                    Bewertung Makrophyten/Diatomeen   
 8   DELW_DEBB800015828419                          Teilbewertung Makrophyten   
 9   DELW_DEBB800015828419  Teilbewertung Diatomeen/Zustands-/Potentialkla...   
 10  DELW_DEBB800015828419                               Ökologischer Zustand   
 11  DELW_DEBB800015828419  

In [81]:
gw = pd.read_csv(data / "GW_measures.csv", low_memory=False)
ow = pd.read_csv(data / "OW_measures.csv", low_memory=False)

In [87]:
def text_column(series):
    return series.astype("string").str.strip()

In [88]:
# Schadstoffe und chemischer Zustand aus GW_measures.csv und OW_measures.csv
def load_chemical_status(gw, ow):
    engine = conn_db()

    codes = pd.concat([gw["POLLCODE"], ow["PSCODE"]])
    pollutants = pd.DataFrame({"pollutant_code": codes.dropna().unique()})
    update_db(pollutants,engine, "pollutant", ["pollutant_code"])

   
    pollutant_table = pd.read_sql_table("pollutant", engine, schema="water_kg")
    pollutant_ids = dict(zip(pollutant_table.pollutant_code, pollutant_table.pollutant_id))
    water_table = pd.read_sql_table("water_body", engine, schema="water_kg")
    lake_ids = dict(zip(water_table.eu_cd_ls, water_table.eu_cd_lw))

    ground = pd.DataFrame({
        "surface_water_body_id": None,
        "groundwater_body_id": text_column(gw["EU_CD_GB"]),
        "pollutant_id": text_column(gw["POLLCODE"]).map(pollutant_ids),
        "failed_standard": gw["POLL_FAIL"].map({"Y": True, "N": False}),
        "risk_code": text_column(gw["POLL_RISK"]),
        "upward_trend_code": text_column(gw["POLLUPWARD"]),
        "trend_code": text_column(gw["POLLTRENDR"]),
        "trend_reversal_code": text_column(gw["POLLTRER_P"]),
        "exemption_type": text_column(gw["EX_CHE_TYP"]),
        "exemption_reason": text_column(gw["EX_CHE_PR"]),
        "exception_code": text_column(gw["POLLEXNTC"]),
        "level_set": text_column(gw["LEVELSET"]),
        "level_value": pd.to_numeric(gw["LEVELVALUE"], errors="coerce"),
        "level_unit": text_column(gw["LEVELUNIT"]),
        "inserted_at": pd.to_datetime(gw["INS_WHEN"], errors="coerce").dt.date,
        "inserted_by": gw["INS_BY"], "river_basin_code": text_column(gw["RBD_CD"]),
        "metadata_reference": gw["METADATA"],
    })
    surface = pd.DataFrame({
        "surface_water_body_id": text_column(ow["EU_CD_LS"]).map(lake_ids),
        "groundwater_body_id": None,
        "pollutant_id": text_column(ow["PSCODE"]).map(pollutant_ids),
        "failed_standard": ow["PS_FAIL"].map({"Y": True, "N": False}),
        "fail_reason_code": text_column(ow["FAILRBSP"]),
        "fail_reason_other": text_column(ow["FAILRBS_OT"]),
        "exemption_type": text_column(ow["CHEMEXTYPE"]),
        "exemption_reason": text_column(ow["CHEMEXPRE"]),
        "exception_code": text_column(ow["PS_EXCTYPE"]),
        "inserted_at": pd.to_datetime(ow["INS_WHEN"], errors="coerce").dt.date,
        "inserted_by": ow["INS_BY"], "river_basin_code": text_column(ow["RBD_CD"]),
        "metadata_reference": ow["METADATA"],
    })
    status = pd.concat([ground, surface], ignore_index=True)
    status = status.dropna(subset=["pollutant_id"])
    status = status[status.surface_water_body_id.notna() | status.groundwater_body_id.notna()]
    return update_db(
        status,engine, "water_body_chemical_status",
        ["surface_water_body_id", "groundwater_body_id", "pollutant_id", "inserted_at"],
    )

In [89]:
load_chemical_status(gw,ow)

Verbindung erfolgreich
No new rows for water_kg.pollutant


c:\Users\lschild\Documents\SOLVE\.venv\Lib\site-packages\pandas\io\sql.py:1747: SAWarning: Did not recognize type 'geometry' of column 'geometry'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


In [99]:
# Schutzgebiete aus den drei Shapefiles
def load_protected_areas():
    engine = conn_db()
    definitions = [
        ("ffh.shp", "EU_CD_PH"),
        ("waterprotection.shp", "EU_CD_PD"),
        ("recreation_areas.shp", "EU_CD_PR"),
    ]
    frames = []
    for filename, id_column in definitions:
        source = gpd.read_file(data / filename).to_crs(25833)
        geometry = source.geometry.map(as_multipolygon)
        frame = gpd.GeoDataFrame({
            "protected_area_id": source[id_column],
            "name": source["NAME"],
            "area_type": source["PROT_TYPE"],
            "legislation_code": source["LEG_CD"],
            #"legislation_name": source["LEGIS_NAME"],
            "legislation_url": source["URL"],
        }, geometry=geometry, crs=source.crs)
        frames.append(frame)
    areas = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=25833)
    update_db(areas, engine, "protected_area", ["protected_area_id"], geo=True)

    source = pd.read_csv(data / "protection_area_bewertung.csv", low_memory=False)
    engine = conn_db()
    valid_water = set(pd.read_sql("SELECT eu_cd_lw FROM water_kg.water_body", engine)["eu_cd_lw"])
    valid_areas = set(pd.read_sql("SELECT protected_area_id FROM water_kg.protected_area", engine)["protected_area_id"])
    relations = pd.DataFrame({
        "water_body_id": source["EU_CD_WB"],
        "protected_area_id": source["EU_CD_P"],
        "relation_method": "source_reference",
        "evolution_type": source["EVOLUTIONT"],
        "exemption_code": source["PAREA_EX"],
        "comment": source["PA_COMMENT"],
    })
    relations = relations[
        relations.water_body_id.isin(valid_water)
        & relations.protected_area_id.isin(valid_areas)
    ]
    update_db(
        relations, engine, "water_body_protected_area",
        ["water_body_id", "protected_area_id", "relation_method"],
    )
    return areas, relations

In [100]:
load_protected_areas()

Verbindung erfolgreich
Verbindung erfolgreich


(   protected_area_id                                         name area_type  \
 0          DE3542305                     Mittlere Havel Ergänzung         H   
 1          DE2847304              Platkowsee-Netzowsee-Metzelthin         H   
 2          DE3951302                            Alte Spreemündung         H   
 3          DE3951305                      Uferwiesen bei Niewisch         H   
 4          DE4051302                     Dobberburger Mühlenfließ         H   
 ..               ...                                          ...       ...   
 66      DEPR_BB_0090  SCHARMÜTZELSEE, BAD SAAROW, STRANDBAD MITTE         R   
 67      DEPR_BB_0094   SCHARMÜTZELSEE, WENDISCH RIETZ, FERIENPARK         R   
 68      DEPR_BB_0096         SCHWIELOCHSEE, CP TREBATSCH -  SAWAL         R   
 69      DEPR_BB_0097                      SCHWIELOCHSEE, NIEWISCH         R   
 70      DEPR_BB_0243        TEMPLINER SEE, TEMPLIN, SCHINDERKUHLE         R   
 
    legislation_code legislation_url  

In [101]:
# Raeumlich abgeleitete Beziehungen direkt mit PostGIS erzeugen
def load_spatial_relations():
    engine = conn_db()
    statements = [
        """INSERT INTO water_kg.station_groundwater_body
        (station_id, groundwater_body_id, relation_type, determination_method)
        SELECT s.station_id, g.eu_cd_gb, 'located_in', 'spatial_join'
        FROM water_kg.monitoring_station s JOIN water_kg.groundwater_body g
        ON s.location IS NOT NULL AND ST_Intersects(s.location, g.geometry)
        ON CONFLICT (station_id, groundwater_body_id) DO NOTHING""",
        """INSERT INTO water_kg.water_body_groundwater_body
        (water_body_id, groundwater_body_id, relation_type, overlap_area_km2,
         water_body_overlap_ratio, determination_method)
        SELECT w.eu_cd_lw, g.eu_cd_gb, 'overlaps',
        ST_Area(ST_Intersection(w.geometry,g.geometry))/1000000.0,
        ST_Area(ST_Intersection(w.geometry,g.geometry))/NULLIF(ST_Area(w.geometry),0),
        'spatial_join' FROM water_kg.water_body w JOIN water_kg.groundwater_body g
        ON ST_Intersects(w.geometry,g.geometry)
        ON CONFLICT (water_body_id, groundwater_body_id) DO NOTHING""",
        """INSERT INTO water_kg.catchment_groundwater_body
        (catchment_id, groundwater_body_id, relation_type, overlap_area_km2,
         catchment_overlap_ratio, determination_method)
        SELECT c.catchment_id, g.eu_cd_gb, 'overlaps',
        ST_Area(ST_Intersection(c.geometry,g.geometry))/1000000.0,
        ST_Area(ST_Intersection(c.geometry,g.geometry))/NULLIF(ST_Area(c.geometry),0),
        'spatial_join' FROM water_kg.catchment c JOIN water_kg.groundwater_body g
        ON ST_Intersects(c.geometry,g.geometry)
        ON CONFLICT (catchment_id, groundwater_body_id) DO NOTHING""",
        """INSERT INTO water_kg.station_catchment
        (station_id, catchment_id, relation_type, determination_method)
        SELECT s.station_id, c.catchment_id, 'located_in', 'spatial_join'
        FROM water_kg.monitoring_station s JOIN water_kg.catchment c
        ON s.location IS NOT NULL AND ST_Intersects(s.location, c.geometry)
        ON CONFLICT (station_id, catchment_id) DO NOTHING""",
    ]
    with engine.begin() as connection:
        for statement in statements:
            connection.exec_driver_sql(statement)
    print("Raeumliche Beziehungen wurden aktualisiert.")

In [102]:
load_spatial_relations()

Verbindung erfolgreich


ProgrammingError: (psycopg.errors.UndefinedTable) Relation »water_kg.station_groundwater_body« existiert nicht
LINE 1: INSERT INTO water_kg.station_groundwater_body
                    ^
[SQL: INSERT INTO water_kg.station_groundwater_body
        (station_id, groundwater_body_id, relation_type, determination_method)
        SELECT s.station_id, g.eu_cd_gb, 'located_in', 'spatial_join'
        FROM water_kg.monitoring_station s JOIN water_kg.groundwater_body g
        ON s.location IS NOT NULL AND ST_Intersects(s.location, g.geometry)
        ON CONFLICT (station_id, groundwater_body_id) DO NOTHING]
(Background on this error at: https://sqlalche.me/e/20/f405)